# 10 — Attivazioni SAE sul test set

Notebook per il calcolo delle attivazioni del MSAE_RW (expansion factor 8, latent_dim=6144) sul **test set** di embedding CLIP ViT-L/14.

**Cosa fa**: carica il checkpoint MSAE_RW addestrato, esegue inferenza in batch sugli embedding CLIP pre-calcolati del test set (`places365_ViT-L~14_test_image_{N_test}_768.npy`) e salva le attivazioni latenti `[N_test, 6144]` come raw memmap float32 per la fase di concept naming.(sostanzialmente fa inferenza col modello sul test set per vedere quali concetti si sono attivati)

**Punto critico — preprocessing**: il checkpoint `.pth` contiene `mean_center`, `scaling_factor`, `target_norm` calcolati sul training set. La classe `SAE` li carica automaticamente e li applica in `preprocess()` invocato da `encode()`. Quindi:
- `SAEDataset` viene chiamato con `mean_center=False, target_norm=0.0` → restituisce gli embedding grezzi.
- `model.encode(x)` chiama internamente `self.preprocess(x)` → applica `(x - mean) * scaling_factor`.
- Non applicare preprocessing manualmente, non ricalcolare le statistiche sul test set.


## 1. Setup — Drive, repo, dipendenze

Monta Google Drive, clona la repo MSAE in `/content/MSAE` (idempotente) e la aggiunge a `sys.path`. Installa `openai-clip` se mancante (importato indirettamente da alcuni moduli della repo).

In [ ]:
import os
import sys
import subprocess
import pathlib

from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = pathlib.Path('/content/MSAE')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/WolodjaZ/MSAE.git', str(REPO_DIR)], check=True)
else:
    print(f'Repo gia\' presente: {REPO_DIR}')

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

try:
    import clip  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'openai-clip'], check=True)
    import clip  # noqa: F401

print(f'cwd = {os.getcwd()}')
print(f'sys.path[0] = {sys.path[0]}')

## 2. Configurazione path

Tutte le variabili configurabili sono qui in cima. Nessun path hardcoded nel resto del notebook.

Gli embedding CLIP (train/test) hanno naming `places365_ViT-L~14_{split}_image_{N}_768.npy` con N che dipende dallo split stratificato. Li cerco con glob in `DRIVE_ROOT` e nelle sottocartelle `outputs/`, `data/` (stessa strategia del notebook 7) — cosi' funziona indipendentemente da dove sono stati caricati su Drive.

In [ ]:
import glob

DRIVE_ROOT = '/content/drive/MyDrive/tesi'

# Checkpoint MSAE_RW (expansion 8, latent 6144) — naming convention della repo.
SAE_CHECKPOINT_PATH = f'{DRIVE_ROOT}/models/6144_768_TopKReLU_64_RW_False_False_0.0_places365_ViT-L~14_train_image_20475_768.pth'

# Directory di output per le attivazioni SAE.
OUTPUT_DIR = f'{DRIVE_ROOT}'

# Numero atteso di immagini test (deve combaciare con il filename TEST_EMB_PATH risolto).
N_TEST = 2938

BATCH_SIZE = 256

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Risoluzione path embedding tramite glob ---
SEARCH_DIRS = [DRIVE_ROOT]

def find_embedding(split: str) -> str:
    pattern = f'places365_ViT-L~14_{split}_image_*_768.npy'
    candidates = []
    for d in SEARCH_DIRS:
        candidates.extend(glob.glob(os.path.join(d, pattern)))
    if not candidates:
        raise FileNotFoundError(
            f'Nessun file embedding per split={split!r} trovato.'
            f'Pattern: {pattern}'
            f'Directory cercate: {SEARCH_DIRS}'
        )
    if len(candidates) > 1:
        print(f'WARNING: trovati {len(candidates)} file per split={split!r}, uso il primo:')
        for c in candidates:
            print(f'  - {c}')
    return candidates[0]

TRAIN_EMB_PATH = find_embedding('train')
TEST_EMB_PATH = find_embedding('test')

print(f'SAE_CHECKPOINT_PATH : {SAE_CHECKPOINT_PATH}')
print(f'TRAIN_EMB_PATH      : {TRAIN_EMB_PATH}')
print(f'TEST_EMB_PATH       : {TEST_EMB_PATH}')

assert os.path.exists(SAE_CHECKPOINT_PATH), f'Checkpoint mancante: {SAE_CHECKPOINT_PATH}'
print('Tutti i file richiesti sono presenti.')

## 3. Caricamento modello SAE

Usa la classe `SAE` di `sae.py`: il filename del checkpoint codifica gli iperparametri (`n_latents=6144`, `n_inputs=768`, `activation=TopKReLU_64`, ecc.) e `load_model` li parsa automaticamente. Vengono caricati anche `mean_center`, `scaling_factor`, `target_norm` salvati durante il training.



In [ ]:
import torch
from sae import SAE

_orig_torch_load = torch.load
def _torch_load_compat(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_torch_load(*args, **kwargs)
torch.load = _torch_load_compat

try:
    model = SAE(SAE_CHECKPOINT_PATH)
finally:
    torch.load = _orig_torch_load

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f'device                 : {device}')
print(f'model.input_dim        : {model.input_dim}')
print(f'model.latent_dim       : {model.latent_dim}')
print(f'model.mean.shape       : {tuple(model.mean.shape)}')
print(f'model.mean.device      : {model.mean.device}')
print(f'model.scaling_factor   : {float(model.scaling_factor):.6f}')

## 4. Caricamento embedding test e sanity check distribuzione

Carico il file `.npy` come memmap tramite `SAEDataset(..., mean_center=False, target_norm=0.0)`. Il dataset restituisce gli embedding **grezzi**, senza modifiche: il preprocessing (centering + scaling con statistiche del training) e' gestito da `SAE.preprocess()` invocato internamente da `SAE.encode()`.

**Sanity check**: confronto la norma L2 media degli embedding di test con quella del training, su un campione di 500 elementi ciascuno. Se i due set hanno subito la stessa pipeline di preprocessing (centering + L2-norm a `target_norm` durante l'estrazione CLIP), le norme medie devono essere comparabili (differenza relativa < 5%). Differenze significative indicano un mismatch di preprocessing a monte.

In [ ]:
import numpy as np
from utils import SAEDataset

dataset_test = SAEDataset(TEST_EMB_PATH, mean_center=False, target_norm=0.0)
dataset_train = SAEDataset(TRAIN_EMB_PATH, mean_center=False, target_norm=0.0)

first = dataset_test[0]
print(f'len(dataset_test)  = {len(dataset_test)}')
print(f'len(dataset_train) = {len(dataset_train)}')
print(f'dataset_test[0].shape = {tuple(first.shape)}  dtype={first.dtype}')

# Verifica coerenza con N_TEST atteso
assert len(dataset_test) == N_TEST, f'Mismatch: dataset_test ha {len(dataset_test)} elementi, atteso {N_TEST}'

# --- Sanity check distribuzione: norma L2 media su campione di 500 ---
rng = np.random.default_rng(42)
n_sample = 500

idx_test = rng.choice(len(dataset_test), size=min(n_sample, len(dataset_test)), replace=False)
idx_train = rng.choice(len(dataset_train), size=min(n_sample, len(dataset_train)), replace=False)

norms_test = np.array([float(torch.linalg.norm(dataset_test[int(i)])) for i in idx_test])
norms_train = np.array([float(torch.linalg.norm(dataset_train[int(i)])) for i in idx_train])

mean_test = float(norms_test.mean())
mean_train = float(norms_train.mean())
rel_diff = abs(mean_test - mean_train) / max(mean_train, 1e-12)

print('\n=== Sanity check distribuzione ===')
print(f'  norma media train (n={len(idx_train)}) : {mean_train:.4f}  std={norms_train.std():.4f}')
print(f'  norma media test  (n={len(idx_test)})  : {mean_test:.4f}  std={norms_test.std():.4f}')
print(f'  differenza relativa                    : {rel_diff*100:.2f}%')

if rel_diff > 0.05:
    print('\n  WARNING: differenza relativa > 5%. Verifica che training e test set abbiano subito')
    print('           lo stesso preprocessing (es. L2-norm a target_norm, centering, scaling).')
else:
    print('\n  OK: distribuzioni di training e test compatibili.')

## 5. Inferenza SAE

DataLoader con `batch_size=BATCH_SIZE`, `shuffle=False` (preserva l'ordine per match con i path immagine), `num_workers=2`. Itera in `torch.no_grad()` con `tqdm`.

`model.encode(x)` restituisce la tupla `(latents_sparse, full_latents)`:
- `latents_sparse` ha il vincolo TopK applicato (solo i K=64 valori piu' grandi non-zero).
- `full_latents` contiene le attivazioni dopo ReLU **senza** vincolo TopK: questo e' quanto serve per l'analisi downstream 

Il preprocessing `(x - mean) * scaling_factor` viene applicato internamente da `model.encode()` tramite `self.preprocess(x)`.

In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
#iteratore che produce batch di 256 embedding CLIP grezzi (non normalizzati), preservando l'ordine originale
loader = DataLoader(
    dataset_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == 'cuda'),
)
#loop di inferenza 
activations_chunks = []
with torch.no_grad():
    for batch in tqdm(loader, desc='SAE inference'):
        batch = batch.to(device, non_blocking=True)
        # encode applica internamente preprocess(): (x - mean) * scaling_factor
        _, full_latents = model.encode(batch)
        activations_chunks.append(full_latents.detach().cpu().numpy().astype(np.float32))
# Concatena tutti i batch insieme
activations = np.concatenate(activations_chunks, axis=0)
del activations_chunks

print(f'activations.shape = {activations.shape}  dtype={activations.dtype}')
assert activations.shape == (N_TEST, model.latent_dim), \
    f'Shape inattesa: {activations.shape}, atteso ({N_TEST}, {model.latent_dim})'

## 6. Salvataggio attivazioni come raw memmap

Salvataggio coerente con la convenzione del resto della pipeline (file `.npy` raw memmap float32, naming che codifica split + numero esempi + dimensione latente).

In [ ]:
output_path = os.path.join(OUTPUT_DIR, f'sae_activations_test_{N_TEST}_{model.latent_dim}.npy')

mm = np.memmap(output_path, dtype=np.float32, mode='w+', shape=activations.shape)
mm[:] = activations
mm.flush()
del mm

print(f'Salvato: {output_path}')
print(f'Size:    {os.path.getsize(output_path) / 1e6:.1f} MB')

## 7. Sanity check obbligatorio

Ricarica il file da disco come memmap e verifica:
- shape e dtype corrispondono a quanto atteso (`[N_TEST, 6144]`, `float32`)
- sparsity media nell'intervallo atteso per MSAE RW (~0.80–0.85)
- numero di neuroni mai attivi (dead neurons)
- statistiche sui valori delle attivazioni

Se la sparsity esce dall'intervallo `[0.75, 0.90]` stampa un warning esplicito.

In [ ]:
acts = np.memmap(output_path, dtype=np.float32, mode='r', shape=(N_TEST, model.latent_dim))

assert acts.shape == (N_TEST, model.latent_dim), f'Shape errata: {acts.shape}'
assert acts.dtype == np.float32, f'Dtype errato: {acts.dtype}'

sparsity = float((acts == 0).mean())
dead_neurons = int((acts == 0).all(axis=0).sum())
active_per_image = float((acts != 0).sum(axis=1).mean())
max_val = float(acts.max())
nonzero_mask = acts > 0
min_nonzero = float(acts[nonzero_mask].min()) if nonzero_mask.any() else float('nan')

print(f'Shape                       : {acts.shape}')
print(f'Dtype                       : {acts.dtype}')
print(f'Sparsita\' media             : {sparsity:.4f}   (atteso ~0.80-0.85 per MSAE RW)')
print(f'Neuroni mai attivi          : {dead_neurons} / {model.latent_dim}')
print(f'Neuroni attivi medi/immagine: {active_per_image:.1f}')
print(f'Valore max attivazione      : {max_val:.4f}')
print(f'Valore min non-zero         : {min_nonzero:.6f}')

if not (0.75 <= sparsity <= 0.90):
    print('\n  WARNING: sparsity fuori dall\'intervallo atteso [0.75, 0.90].')
    print('           Verifica che il preprocessing sia stato applicato correttamente')
    print('           (centering + scaling con le statistiche del training).')
else:
    print('\n  OK: sparsity nell\'intervallo atteso.')